# **HANDS - ON: FINE-TUNING CON LORA**

Una vez vista la masterclass ***Fine-tuning y Evaluación de Modelos***, se proporciona el siguiente ***Colab*** para ejecutar, en vivo, un fine-tuning real con LoRA sobre un modelo Llama ligero, y medir su mejora con una métrica objetiva.

A diferencia de los Temas anteriores, aquí no usamos Groq — Groq solo sirve para inferencia, no para entrenar modelos. Usamos **Hugging Face** (librerías `transformers` y `peft`) directamente sobre la GPU gratuita de Colab.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1cKZ_hCf231RE84FDvGkEiKMH6ZDkVZzH?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura, añadiendo un nombre asociado al token para guardarlo dentro de una variable y usarlo dentro del notebook. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [12]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets
import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("Sesión de Hugging Face iniciada correctamente.")


Sesión de Hugging Face iniciada correctamente.


### **CARGAR EL MODELO BASE**

Usamos un modelo Llama ligero (pocos parámetros) para que el fine-tuning corra en minutos sobre la GPU T4 gratuita de Colab, sin necesitar cuantización adicional.

In [16]:
# Cargar el modelo base de Llama y su tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


### **ANTES DEL FINE-TUNING: LÍNEA BASE**

Antes de ajustar nada, probamos el modelo base con un prompt de ejemplo para tener un punto de comparación. El modelo aún no conoce el tono ni el formato que le vamos a enseñar.

In [17]:
# Definir una función para generar texto y probar el modelo base con un prompt de ejemplo

def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):
    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado.split("\n")[0].strip()

prompt_prueba = "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente:"

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)


No, no puedes cambiar tu pedido.


### **PREPARAR LOS DATOS DE ENTRENAMIENTO**

El fine-tuning necesita ejemplos de entrada y salida que muestren el comportamiento que queremos enseñarle al modelo. Con pocos ejemplos (5 a 10) es suficiente para una demo — no es un dataset de producción.

In [ ]:
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset

from datasets import Dataset

ejemplos = [
    {"texto": "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente: Sí, puedes "
     "solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com."},
    {"texto": "Cliente: ¿Cuánto tarda el reembolso?\nAgente: El reembolso se refleja en un plazo de 5 a 7 días hábiles."},
    {"texto": "Cliente: ¿Tienen envío el mismo día?\nAgente: Sí, disponible en zonas seleccionadas si el pedido se confirma antes de las 12:00."},
    {"texto": "Cliente: ¿Puedo pagar en el momento de la entrega?\nAgente: Sí, aceptamos pago contra entrega en efectivo o tarjeta."},
    {"texto": "Cliente: ¿Cómo rastreo mi paquete?\nAgente: Puedes rastrearlo con el número de guía en la sección 'Mis pedidos' de tu cuenta."},
]

dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

## **REALIZAR FINE-TUNING**

### **CONFIGURAR Y APLICAR LORA**

LoRA agrega matrices pequeñas entrenables sin tocar los pesos originales del modelo — por eso es tan ligero comparado con un fine-tuning completo.

In [ ]:
# Configurar LoRA (rango, alpha, módulos objetivo) y aplicarlo al modelo base


### **ENTRENAR CON LORA**

Con el dataset y LoRA ya configurados, ejecutamos el entrenamiento. La pérdida (*loss*) que reporta el entrenador es nuestra métrica objetiva: debería bajar a medida que el modelo aprende los ejemplos.

In [ ]:
# Configurar el entrenador (SFTTrainer) y ejecutar el fine-tuning


### **DESPUÉS DEL FINE-TUNING: MEDIR LA MEJORA**

Compararemos la pérdida antes y después del entrenamiento como métrica objetiva.

In [ ]:
# Comparar la pérdidas


In [ ]:
# Referencia cualitativa vs Referencia con un modelo de producción


**Nota:** Por eso, además de la respuesta de TinyLlama, se muestra una respuesta de Groq (GPT-OSS-20B) para el mismo prompt: no porque el proceso de fine-tuning haya fallado, sino como punto de comparación de cómo respondería un modelo de producción con muchos más parámetros y datos de entrenamiento — TinyLlama con 5 ejemplos demuestra la técnica de LoRA, no busca igualar la calidad de un modelo así de grande.

## **CHALLENGE: AJUSTE DE TONO CON LORA**

Una vez visto el ***Hands-On: Fine-tuning con LoRA***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se ajustará un modelo Llama ligero con un dataset propio para enseñarle un tono o formato de respuesta específico, comparando la pérdida **antes** y **después** del fine-tuning como métrica objetiva. En esta solución se usa como ejemplo un asistente de dudas frecuentes del propio curso.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Carga el modelo y define tu dataset:**

   * Instala las librerías, inicia sesión en Hugging Face con tu token y carga el modelo base junto con la función `generar_respuesta`.

   * Construye una lista llamada `ejemplos` con al menos 4 pares de entrada/respuesta que reflejen el tono o formato que quieres enseñarle al modelo, y conviértela en `dataset`.

In [1]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets

!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("Sesión de Hugging Face iniciada correctamente.")


Sesión de Hugging Face iniciada correctamente.


In [27]:
# Cargar el modelo base de Llama y su tokenizer

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [28]:
# Definir la funcion de generacion de texto

def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=512):
    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        min_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado.split("\n")[0].strip()

prompt_prueba = "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente:"

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)



No, no puedes cambiar tu pedido.


In [29]:
# Definir la lista ejemplos y convertirla en dataset

from datasets import Dataset
ejemplos1 = [
    {"texto": "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente: Sí, puedes "
     "solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com."},
    {"texto": "Cliente: ¿Cuánto tarda el reembolso?\nAgente: El reembolso se refleja en un plazo de 5 a 7 días hábiles."},
    {"texto": "Cliente: ¿Tienen envío el mismo día?\nAgente: Sí, disponible en zonas seleccionadas si el pedido se confirma antes de las 12:00."},
    {"texto": "Cliente: ¿Puedo pagar en el momento de la entrega?\nAgente: Sí, aceptamos pago contra entrega en efectivo o tarjeta."},
    {"texto": "Cliente: ¿Cómo rastreo mi paquete?\nAgente: Puedes rastrearlo con el número de guía en la sección 'Mis pedidos' de tu cuenta."},
]

ejemplos2 = [
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi computadora no enciende."},
            {"role": "assistant", "content": "Verifica que el cable de corriente esté conectado correctamente y prueba con otro enchufe. Si no enciende, puede ser necesario revisar la fuente de poder."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi PC enciende pero no aparece imagen."},
            {"role": "assistant", "content": "Verifica que el monitor esté encendido y que el cable HDMI o DisplayPort esté conectado correctamente. También puedes probar otro puerto de video."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "La computadora se apaga sola después de unos minutos."},
            {"role": "assistant", "content": "Puede deberse a sobrecalentamiento, problemas de alimentación o fallas de hardware. Revisa que los ventiladores estén funcionando y que las entradas de aire no estén obstruidas."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi computadora está muy lenta."},
            {"role": "assistant", "content": "Revisa cuánto espacio libre tienes en el disco y qué programas se ejecutan al iniciar Windows. También conviene comprobar el uso de CPU y memoria desde el Administrador de tareas."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Windows tarda mucho en iniciar."},
            {"role": "assistant", "content": "Desactiva los programas innecesarios que se ejecutan al iniciar Windows desde el Administrador de tareas. También revisa el estado del disco y el espacio disponible."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi computadora se congela constantemente."},
            {"role": "assistant", "content": "Puede existir un problema de memoria RAM, almacenamiento, temperatura o algún controlador. Ejecuta un diagnóstico de memoria y revisa las temperaturas del sistema."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo puedo reiniciar Windows en modo seguro?"},
            {"role": "assistant", "content": "Desde Windows puedes acceder a Configuración, Sistema, Recuperación y después seleccionar Inicio avanzado para reiniciar y entrar a las opciones de modo seguro."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo puedo saber cuánta memoria RAM tiene mi PC?"},
            {"role": "assistant", "content": "Abre Configuración, Sistema y Acerca de. En esa sección encontrarás la cantidad de memoria RAM instalada."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo reviso qué procesador tiene mi computadora?"},
            {"role": "assistant", "content": "Puedes verlo en Configuración, Sistema y Acerca de. También puedes abrir el Administrador de tareas y consultar la información en la pestaña Rendimiento."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi disco duro está casi lleno."},
            {"role": "assistant", "content": "Revisa Configuración, Sistema, Almacenamiento para identificar qué archivos y aplicaciones están ocupando más espacio. Puedes eliminar archivos temporales y aplicaciones que ya no utilices."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo libero espacio en Windows?"},
            {"role": "assistant", "content": "Puedes utilizar la herramienta de limpieza de almacenamiento de Windows para eliminar archivos temporales, archivos de actualización y otros datos que ya no sean necesarios."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi teclado no funciona."},
            {"role": "assistant", "content": "Desconecta y vuelve a conectar el teclado. Si es USB, prueba otro puerto. Si es inalámbrico, revisa las baterías y vuelve a conectar el receptor o dispositivo Bluetooth."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Algunas teclas de mi teclado no funcionan."},
            {"role": "assistant", "content": "Prueba el teclado en otra computadora para determinar si el problema es del dispositivo. Si el problema ocurre únicamente en tu PC, revisa el controlador y la configuración del teclado."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi mouse dejó de funcionar."},
            {"role": "assistant", "content": "Si es USB, prueba otro puerto. Si es inalámbrico, cambia las baterías y verifica que el receptor esté conectado. También puedes probar el mouse en otra computadora."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "El touchpad de mi laptop no funciona."},
            {"role": "assistant", "content": "Verifica que el touchpad no haya sido desactivado mediante una tecla de función. Si continúa sin funcionar, revisa su configuración y el controlador correspondiente en Windows."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "No tengo conexión a Internet."},
            {"role": "assistant", "content": "Comprueba si otros dispositivos tienen conexión. Si ninguno tiene Internet, reinicia el router. Si solamente falla la computadora, desactiva y vuelve a activar el adaptador de red."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "El Wi-Fi aparece pero no puedo conectarme."},
            {"role": "assistant", "content": "Comprueba que la contraseña sea correcta y olvida la red desde la configuración de Wi-Fi para conectarte nuevamente. También puedes reiniciar el adaptador de red."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi Internet funciona muy lento en la computadora."},
            {"role": "assistant", "content": "Comprueba la velocidad en otro dispositivo y verifica la intensidad de la señal Wi-Fi. Si otros dispositivos funcionan correctamente, revisa el adaptador de red y los programas que estén utilizando Internet."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo puedo conectarme a una red Wi-Fi?"},
            {"role": "assistant", "content": "Selecciona el icono de red en la barra de tareas, elige la red Wi-Fi que deseas utilizar e introduce la contraseña cuando Windows la solicite."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "No funciona el Bluetooth de mi PC."},
            {"role": "assistant", "content": "Verifica que Bluetooth esté activado en Configuración. Si está activado pero no detecta dispositivos, reinicia Bluetooth y comprueba que el controlador del adaptador esté actualizado."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "No escucho ningún sonido."},
            {"role": "assistant", "content": "Comprueba que el volumen no esté silenciado y que Windows tenga seleccionado el dispositivo de salida correcto. También puedes revisar el controlador de audio."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Los audífonos no funcionan."},
            {"role": "assistant", "content": "Verifica que estén correctamente conectados y que Windows los haya seleccionado como dispositivo de salida. Si son Bluetooth, comprueba que estén conectados y tengan batería."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "El micrófono no funciona."},
            {"role": "assistant", "content": "Revisa que Windows tenga seleccionado el micrófono correcto y que las aplicaciones tengan permiso para utilizarlo. También puedes probar el micrófono en otra aplicación."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "La pantalla se ve demasiado oscura."},
            {"role": "assistant", "content": "Comprueba el nivel de brillo de Windows y los controles del monitor. Si el brillo cambia inesperadamente, revisa también la configuración de ahorro de energía."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "La pantalla parpadea constantemente."},
            {"role": "assistant", "content": "Comprueba el cable de video y prueba otro puerto. También revisa la frecuencia de actualización configurada y actualiza el controlador gráfico si es necesario."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Windows muestra una pantalla azul."},
            {"role": "assistant", "content": "Una pantalla azul generalmente indica un problema grave de hardware, controladores o software. Anota el código del error y revisa los controladores instalados recientemente."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Windows no encuentra mi impresora."},
            {"role": "assistant", "content": "Comprueba que la impresora esté encendida y conectada a la misma red que la computadora. Si es USB, prueba otro puerto y verifica que Windows detecte el dispositivo."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "La impresora aparece desconectada."},
            {"role": "assistant", "content": "Verifica la conexión USB o de red y asegúrate de que la impresora esté encendida. Desde Windows también puedes revisar la cola de impresión y reiniciar el servicio de impresión."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "No puedo imprimir un documento."},
            {"role": "assistant", "content": "Comprueba que la impresora esté seleccionada correctamente y que no existan documentos detenidos en la cola de impresión. Si es necesario, cancela los trabajos pendientes y vuelve a imprimir."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Windows no reconoce una memoria USB."},
            {"role": "assistant", "content": "Prueba la memoria USB en otro puerto y, si es posible, en otra computadora. Si el dispositivo funciona en otro equipo, revisa Administración de discos y los controladores USB de Windows."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi memoria USB pide ser formateada."},
            {"role": "assistant", "content": "No la formatees todavía si necesitas recuperar los archivos. El mensaje puede indicar un problema con el sistema de archivos o con la memoria. Primero intenta acceder a ella desde otra computadora."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo puedo hacer una copia de seguridad de mis archivos?"},
            {"role": "assistant", "content": "Puedes copiar tus archivos importantes a un disco externo o utilizar un servicio de almacenamiento en la nube. Lo recomendable es mantener más de una copia de los archivos importantes."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Eliminé un archivo por accidente."},
            {"role": "assistant", "content": "Revisa primero la Papelera de reciclaje. Si el archivo no está allí, evita guardar nuevos datos en ese disco hasta determinar si será necesario utilizar una herramienta de recuperación."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi antivirus detectó un virus."},
            {"role": "assistant", "content": "No abras ni ejecutes el archivo detectado. Permite que Windows Security o tu antivirus lo ponga en cuarentena y realiza un análisis completo del sistema."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Creo que mi computadora tiene malware."},
            {"role": "assistant", "content": "Ejecuta un análisis completo con Windows Security o un antivirus confiable. Si aparecen ventanas extrañas o programas desconocidos, evita introducir contraseñas hasta comprobar que el sistema esté limpio."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo actualizo Windows?"},
            {"role": "assistant", "content": "Abre Configuración, Windows Update y selecciona Buscar actualizaciones. Instala las actualizaciones disponibles y reinicia la computadora si Windows lo solicita."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Debo actualizar los controladores de mi computadora?"},
            {"role": "assistant", "content": "Los controladores pueden actualizarse cuando existe un problema de compatibilidad, seguridad o funcionamiento. Para componentes importantes, utiliza preferentemente Windows Update o el sitio oficial del fabricante."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi computadora se calienta demasiado."},
            {"role": "assistant", "content": "Revisa que las entradas y salidas de aire estén libres de polvo y que los ventiladores funcionen correctamente. También conviene revisar las temperaturas del procesador y la tarjeta gráfica."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "El ventilador de mi computadora hace mucho ruido."},
            {"role": "assistant", "content": "El ruido puede aumentar cuando el procesador o la tarjeta gráfica están trabajando intensamente. Revisa las temperaturas y asegúrate de que los ventiladores y las entradas de aire estén libres de polvo."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo puedo saber qué programa está haciendo lenta mi computadora?"},
            {"role": "assistant", "content": "Abre el Administrador de tareas y revisa el consumo de CPU, memoria, disco y red. Ordena las columnas para identificar qué procesos están utilizando más recursos."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Una aplicación de Windows no responde."},
            {"role": "assistant", "content": "Espera unos segundos y, si continúa bloqueada, abre el Administrador de tareas, selecciona la aplicación y utiliza Finalizar tarea. Si ocurre frecuentemente, revisa si existen actualizaciones para la aplicación."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo desinstalo un programa?"},
            {"role": "assistant", "content": "Abre Configuración, Aplicaciones y Aplicaciones instaladas. Busca el programa que deseas eliminar y selecciona Desinstalar."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Instalé un programa y ahora Windows funciona mal."},
            {"role": "assistant", "content": "Si el problema comenzó inmediatamente después de instalar el programa, intenta desinstalarlo. También puedes utilizar un punto de restauración si el problema persiste."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo puedo restaurar Windows a un punto anterior?"},
            {"role": "assistant", "content": "Puedes utilizar Restaurar sistema si existen puntos de restauración disponibles. Esta función permite regresar configuraciones y archivos del sistema a un estado anterior sin afectar normalmente tus documentos personales."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi laptop no carga la batería."},
            {"role": "assistant", "content": "Comprueba que el cargador esté conectado correctamente y que el indicador de carga esté encendido. Si es posible, prueba con otro cargador compatible y revisa el estado de la batería en Windows."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "La batería de mi laptop dura muy poco."},
            {"role": "assistant", "content": "Revisa qué aplicaciones están consumiendo más energía y utiliza el modo de ahorro de batería. Si la autonomía ha disminuido considerablemente con el tiempo, la batería puede estar degradada."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo puedo saber si mi disco SSD está fallando?"},
            {"role": "assistant", "content": "Puedes revisar su estado mediante herramientas de diagnóstico del fabricante y observar si Windows presenta errores frecuentes, bloqueos o problemas al leer y escribir archivos."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo entro al BIOS de mi computadora?"},
            {"role": "assistant", "content": "Reinicia la computadora y presiona repetidamente la tecla indicada por el fabricante durante el arranque. Las teclas más comunes son F2, F10, F12, Delete o Esc."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Quiero instalar Windows desde una USB."},
            {"role": "assistant", "content": "Necesitas una memoria USB booteable con el instalador de Windows. Después debes configurar el equipo para arrancar desde USB y seguir las instrucciones del instalador."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Mi computadora no arranca después de una actualización de Windows."},
            {"role": "assistant", "content": "Intenta entrar en el entorno de recuperación de Windows y utiliza Reparación de inicio. Si el problema comenzó después de una actualización reciente, también puedes intentar desinstalar esa actualización."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "¿Cómo puedo comprobar si mi RAM está fallando?"},
            {"role": "assistant", "content": "Puedes ejecutar el Diagnóstico de memoria de Windows para buscar errores. Si aparecen problemas, también es recomendable probar los módulos de memoria individualmente cuando el equipo lo permita."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "Quiero conectar un segundo monitor."},
            {"role": "assistant", "content": "Conecta el segundo monitor mediante una salida compatible como HDMI, DisplayPort o USB-C. Después abre Configuración de pantalla para seleccionar si deseas duplicar o extender el escritorio."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Eres un experto en soporte técnico computacional que brinda soluciones concisas y precisas."},
            {"role": "user", "content": "El segundo monitor no aparece en Windows."},
            {"role": "assistant", "content": "Comprueba el cable y la entrada seleccionada en el monitor. En Configuración de pantalla puedes utilizar la opción Detectar para buscar manualmente una pantalla adicional."}
        ]
    }
]


ejemplos = ejemplos2
dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['messages'],
    num_rows: 53
})

**2. Prueba el modelo base:** Genera una respuesta con el modelo sin ajustar para un prompt de prueba y guárdala en `respuesta_base`.

In [30]:
# Probar el modelo base con un prompt de prueba y guardar el resultado en respuesta_base

prompt_prueba = "Cliente: Mi computadora hace un ruido constante muy fuerte siempre que esta encendida. ¿Que puede ser?\nAgente:"

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

No sé, pero puedes probar cambiando el ordenador a modo de modo de desconexión y luego a modo conectado.


**3. Configura y aplica LoRA:** Fija una semilla con `set_seed` y define tu `LoraConfig` (rango, alpha, módulos objetivo, dropout en 0) y aplícalo al modelo base.

In [33]:
# Configurar LoraConfig y aplicarlo al modelo base

!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()
# r más alto = más capacidad para aprender, pero también más parámetros entrenables

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


**4. Entrena:** Configura el `SFTTrainer` con tu dataset y ejecuta el fine-tuning; guarda la pérdida final en `perdida_final`.

In [34]:
# Configurar el Trainer con el dataset y ejecutar el fine-tuning; guardar la pérdida final en perdida_final

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=4,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="messages",
    max_length=2048,
    report_to="none",
    #fp16=True,
    #bf16=False,
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
print("Pérdida final:", resultado_entrenamiento.training_loss)

Tokenizing train dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

{'loss': '2.61', 'grad_norm': '1.779', 'learning_rate': '0.0002', 'entropy': '1.701', 'num_tokens': '215', 'mean_token_accuracy': '0.5352', 'epoch': '0.03704'}
{'loss': '2.783', 'grad_norm': '1.902', 'learning_rate': '0.0001981', 'entropy': '1.795', 'num_tokens': '444', 'mean_token_accuracy': '0.5154', 'epoch': '0.07407'}
{'loss': '2.377', 'grad_norm': '1.908', 'learning_rate': '0.0001963', 'entropy': '1.634', 'num_tokens': '677', 'mean_token_accuracy': '0.5844', 'epoch': '0.1111'}
{'loss': '2.466', 'grad_norm': '2.443', 'learning_rate': '0.0001944', 'entropy': '1.687', 'num_tokens': '881', 'mean_token_accuracy': '0.5842', 'epoch': '0.1481'}
{'loss': '2.205', 'grad_norm': '2.703', 'learning_rate': '0.0001926', 'entropy': '1.752', 'num_tokens': '1089', 'mean_token_accuracy': '0.6165', 'epoch': '0.1852'}
{'loss': '2.346', 'grad_norm': '2.733', 'learning_rate': '0.0001907', 'entropy': '1.603', 'num_tokens': '1311', 'mean_token_accuracy': '0.6', 'epoch': '0.2222'}
{'loss': '2.058', 'grad_n

**5. Compara y concluye:** Calcula la reducción entre la pérdida inicial y `perdida_final` como métrica objetiva, y usa la respuesta generada por el modelo ya ajustado solo como referencia cualitativa.

In [35]:
# Comparar la perdida inicial y final del entrenamiento como metrica objetiva

perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = resultado_entrenamiento.training_loss

print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

# trainer.state.log_history[0]['loss'] es la pérdida después del primer paso registrado,
# no la pérdida real del modelo sin ningún entrenamiento.

Pérdida al inicio del entrenamiento: 2.61
Pérdida final del entrenamiento: 1.16
Reducción: 56%


In [36]:
# Como referencia cualitativa (puede variar de sesión a sesión con un dataset tan chico):
!pip install groq -q
from groq import Groq
import json

# respuesta_ajustada = generar_respuesta(modelo_lora, prompt_prueba)
prompt_prueba1 = "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente:"
prompt_prueba2 =  "Hola que tal"
prompt_prueba = prompt_prueba2
respuesta_ajustada = generar_respuesta(modelo_lora, prompt_prueba)
print("Respuesta del modelo ajustado (TinyLlama + LoRA):\n", respuesta_ajustada)


#client = Groq(api_key=userdata.get('APIK'))
#response = client.chat.completions.create(
#    model="openai/gpt-oss-20b",
#    messages=[{"role": "user", "content": prompt_prueba + " Respuesta muy breve y corta."}]
#)

#print("\nRespuesta de referencia (Groq / GPT-OSS-20B):\n", response.choices[0].message.content)

Respuesta del modelo ajustado (TinyLlama + LoRA):
 , and the most important|||>
